# 01 — Basic Agent

This notebook demonstrates the simplest possible orqest agent: a single agent that takes a user message, runs it through an LLM, and returns structured output.

**Prerequisites:**
- A `.env` file in the project root (or environment variables) with:
  ```
  LLM_API_KEY=your_api_key
  LLM_MODEL=openai:gpt-4o   # or anthropic:claude-sonnet-4-20250514, google:gemini-2.0-flash, etc.
  ```

## 1. Load config

orqest never loads environment variables at import time — you call `load_config()` explicitly.

In [1]:
from orqest import load_config

config = load_config()
print(f"Model:    {config.llm_model}")
print(f"API key:  {config.llm_api_key[:8]}...")

Model:    openai:gpt-4.1
API key:  sk-proj-...


## 2. Define the agent

Every orqest agent needs three things:
1. **A state type** — the input (here we use the built-in `GlobalState`)
2. **An output type** — a Pydantic model defining the structured response
3. **`_run_implementation()`** — the async method where your logic lives

In [ ]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


class SummaryOutput(BaseModel):
    """The agent's structured output."""
    summary: str = Field(description="A concise summary of the user's message")
    key_points: list[str] = Field(description="Bullet points of the main ideas")


class SummaryAgent(BaseAgent[GlobalState, SummaryOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="summary_agent",
            system_prompt=(
                "You are a summarization assistant. Given a user's message, "
                "produce a concise summary and a list of key points."
            ),
            output_type=SummaryOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> SummaryOutput:
        user_message = state.get_latest_message("user")
        result = await self.agent.run(user_message)
        return result.output

## 3. Run the agent

Create a `GlobalState`, add a user message, and call `agent.run()`.

In [9]:
agent = SummaryAgent(model=config.llm_model, api_key=config.llm_api_key)

state = GlobalState()
state.add_message(
    "user",
    "Quantum computing uses qubits instead of classical bits. Unlike bits that are "
    "either 0 or 1, qubits can exist in superposition — representing both states "
    "simultaneously. This allows quantum computers to explore many solutions in "
    "parallel. Combined with entanglement, where qubits become correlated regardless "
    "of distance, quantum computers can solve certain problems exponentially faster "
    "than classical machines. Key applications include cryptography, drug discovery, "
    "optimization problems, and materials science."
)

output = await agent.run(state)

print("Summary:")
print(output.summary)
print("\nKey points:")
for point in output.key_points:
    print(f"  - {point}")

Summary:
Quantum computing utilizes qubits, which can be in multiple states simultaneously, allowing it to solve certain problems much faster than classical computers. Applications include areas like cryptography, drug discovery, and optimization.

Key points:
  - Quantum computing uses qubits rather than classical bits.
  - Qubits can exist in a state of superposition, representing both 0 and 1 simultaneously.
  - Superposition enables quantum computers to process many solutions at once.
  - Quantum entanglement allows qubits to be correlated over any distance.
  - These quantum properties allow for exponential speedup in solving certain problems.
  - Key application areas: cryptography, drug discovery, optimization, materials science.


## What's happening under the hood

1. `load_config()` reads your `.env` and validates that `LLM_API_KEY` is set
2. `SummaryAgent.__init__()` calls `resolve_model()` which maps `"openai:gpt-4o"` to the correct pydantic-ai `Model` + `Provider`
3. On first access to `agent.agent`, a pydantic-ai `Agent` is lazily constructed with the system prompt, output type, and model
4. `agent.run(state)` calls `_run_implementation()` which extracts the user message and runs the underlying pydantic-ai agent
5. pydantic-ai validates the LLM response against `SummaryOutput` and returns typed, structured data